# EspY3 flanking-region homologue search
## 4/11/25

EspY3 has a domain of two pentapeptide repeats (PPRs). This is the only annotated region in Pfam and is conserved across 10,000 proteins. Whilst it is highly conserved and structurally stable, the surrounding flanks evolve more rapidly and often determine function or interactions. Variation in these flanks therefore can signal functional divergence or specialisation among homologous proteins. Phylogenetic analysis of these regions may uncover lineage-specific adaptations and link sequence evolution outside the repeat domain to potential differences in protein function within pathogens. 


In this notebook I aim to:
1. Extract the N- and C-terminal flanks *outside* the PPR domain (165-342.
2. Search BLASTP with each flanking sequence to find homologues
3. Curate hits with CD-HIT to remove redundancy
4. Annotate with Pfam/InterPro to see if any new domains appear in the flanks
5. Build allignments and trees


In [13]:
# 1. Isolating the N- and C- flanks.

# N flank residues 1 - 164
!seqkit subseq -r 1:164 ../../data/raw/espy3.fasta > ../../results/flank-search/flank_N.fasta

# C flank residues 343 - 523
!seqkit subseq -r 343:-1 ../../data/raw/espy3.fasta > ../../results/flank-search/flank_C.fasta

[INFO] create or read FASTA index ...
[INFO] read FASTA index from ../../data/raw/espy3.fasta.seqkit.fai
[INFO]   1 records loaded from ../../data/raw/espy3.fasta.seqkit.fai
[INFO] create or read FASTA index ...
[INFO] read FASTA index from ../../data/raw/espy3.fasta.seqkit.fai
[INFO]   1 records loaded from ../../data/raw/espy3.fasta.seqkit.fai


## 2. Blast search

I used the online NCBI BLAST. I searched each flanking regions [N-flank](../../results/flank-search/flank_N_allignedseq.fasta) and [C-flank](../../results/flank-search/flank_C_allignedseq.fasta) against the refseq_protein database, and set the max target sequences to 5000. All other settings are default blastp.

The output of both flanks shows that the flanking regions are restricted to enterobacterales. This supports a model where this protein evolved from a general PPR scaffold but specialised within enteric pathogens, particularly *E. coli*. Both searches also produce a similar number of allignments (761 and 781).

## 3. CD-HIT to group sequences into related clusters

To asses sequence diversity within the EspY3-related family, I clustered the BLAST hits using CD-HIT at identity thresholds 0.95, 0,8  and 0.6. High identity clustering (0.95) removes highly similar strain variants, whilst moderate clustering groups the proteins into broader subfamilies corresponding to different Enterobacterales genera.

In [10]:
# CD-HIT to group sequences into related clusters

# flanks at 99% identity
!cd-hit -i ../../results/flank-search/blastp_output/flank_C_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit99.faa -c 0.99 -n 5
!cd-hit -i ../../results/flank-search/blastp_output/flank_N_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit99.faa -c 0.99 -n 5

# flanks at 95% identity to remove nearly identical duplicates
!cd-hit -i ../../results/flank-search/blastp_output/flank_C_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit95.faa -c 0.95 -n 5
!cd-hit -i ../../results/flank-search/blastp_output/flank_N_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit95.faa -c 0.95 -n 5

# flanks at 80% identity to see if other Enterobacerale homologues group seperately
!cd-hit -i ../../results/flank-search/blastp_output/flank_C_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit80.faa -c 0.8 -n 5
!cd-hit -i ../../results/flank-search/blastp_output/flank_N_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit80.faa -c 0.8 -n 5

# flanks at 60% identity to test if the protein belongs to a broader PPR superfamily
!cd-hit -i ../../results/flank-search/blastp_output/flank_C_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit60.faa -c 0.6 -n 4
!cd-hit -i ../../results/flank-search/blastp_output/flank_N_allignedseq.fasta -o ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit60.faa -c 0.6 -n 4

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:58:32
Command: cd-hit -i
         ../../results/flank-search/blastp_output/flank_C_allignedseq.fasta
         -o
         ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit99.faa
         -c 0.99 -n 5

Started: Thu Nov  6 11:55:37 2025
                            Output                              
----------------------------------------------------------------
total seq: 761
longest and shortest : 185 and 23
Total letters: 94241
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 16M = 16M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 81M

Table limit with the given memory limit:
Max number of representatives: 2554847
Max number of word counting entries: 89798084

comparing sequences from          0  to        761

      761  finished        141  clusters

Approximated maximum memory consumption: 81M
writing new database
writing clus

In [12]:
# How many clusters are in the CD-HIT output files?
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit99.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit99.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit95.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit95.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit80.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit80.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit60.faa.clstr
!grep -c '^>Cluster' ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit60.faa.clstr

141
67
16
9
2
3
1
2


## What do the CD-HIT outputs mean

At 99% identity, the C flank clusters a lot more (141) than the N-flank (67). This suggests the C-flank is much more evolutionary diverse, and that it evolves at a higher rate without constraints. However they both collapse into 1 or 2 groups at 60% identity, which tells us that these variations are within a small family with a number of subgroups.
At 95% identity, the clustering splits 

In 99% identity clustering, I got 97 clusters for each flank. I will allign these with MAFFT, clean with trimAl, and generate a tree with RAxML-NG

In [13]:
# Make MSA

!mafft --localpair --maxiterate 16 --reorder ../../results/flank-search/cd-hit_output/01_multi_id/flank_N_cdhit99.faa > ../../results/flank-search/mafft/flank_N_cdhit99.mafft.faa
!mafft --localpair --maxiterate 16 --reorder ../../results/flank-search/cd-hit_output/01_multi_id/flank_C_cdhit99.faa > ../../results/flank-search/mafft/flank_C_cdhit99.mafft.faa

outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
   60 / 67
done.

Progressive alignment ... 
STEP    66 /66 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

   60 / 67
Segment   1/  1    1- 166
STEP 002-065-1  identical.   
Converged.

done
dvtditr (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 L-INS-i (Probably most accurate, very slow)
 Ite